**03 | Statistical Feature Engineering for Wearable Signals**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Transform HARTH's 2-second annotated sensor windows into an interpretable feature matrix, using a reusable Python function. Inspect real values and activity-dependent distributions.

**Scope:** One participant (`S023`), no model training, feature selection fitted on labels, or train/test split. Feature differences in one person's recording are descriptive, not evidence of unseen-subject performance.

**01. Imports and project setup**

The functions live in `.py` modules; this notebook calls them and shows outputs. Code cells contain short comments to explain *why* each step is performed.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 1. Find the repository root and register the shared/study source folders.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open this notebook inside the repository.")
    root = root.parent

sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

# 2. Reuse the earlier loading/windowing code and import one new feature function.
from core.data_inspection import inspect_data
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features, AVAILABLE_FEATURES
from harth_data import ACTIVITY_NAMES, ensure_harth_sample, load_harth_subject

print("Repository:", root)
print("Available methods:", AVAILABLE_FEATURES)

**02. Load real HARTH signals**

Reuse `S023.csv` from the previous notebooks. All six channels represent accelerometer measurements in the source units (`g`).

In [ ]:
# 1. Load the public source file (or its existing local cache).
data_dir = root / "data" / "raw" / "harth_v2"
csv_path = ensure_harth_sample(data_dir, subject="S023")
data, channel_names = load_harth_subject(csv_path)

# 2. Inspect source dimensions before processing.
inspect_data(data, name="HARTH: continuous sensor data")
print("Channels:", channel_names)

**03. Recreate the validated windowing protocol**

At 50 Hz, 100 samples represent 2 seconds. A 50-sample stride gives 50% overlap between *candidate* windows. The same strict no-transition/no-gap checks from Notebook 02 apply.

In [ ]:
# 1. Keep the protocol identical to Notebook 02 for a fair comparison.
SAMPLING_RATE = 50
WINDOW_SIZE = 100
STEP_SIZE = 50

# 2. Generate windows only where activity labels and timestamps are consistent.
X_windows, y_windows, window_info = create_signal_windows(
    signals=data[channel_names].to_numpy(),
    labels=data["label"].to_numpy(),
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    timestamps=data["timestamp"],
    sampling_rate=SAMPLING_RATE,
    min_label_purity=1.0,
    subject_id=csv_path.stem,
)
print("Window tensor:", X_windows.shape)
print("Aligned labels:", y_windows.shape)
display(window_info.head())

**04. Extract interpretable statistical and temporal features**

The generic function calculates five features for each channel: **Mean, STD, RMS, Peak-to-Peak, and normalised Waveform Length**. With six channels, each window becomes a row containing 30 features. This is a *per-window deterministic transformation*; scaling and model fitting are not performed.

In [ ]:
# 1. Transform (windows, channels, time) into a pandas feature matrix.
feature_table = extract_signal_features(
    X_windows,
    channel_names=channel_names,
    methods=["mean", "std", "rms", "peak_to_peak", "waveform_length"],
)

# 2. Display shape and first rows to make the feature engineering transparent.
feature_report = inspect_data(feature_table, name="HARTH statistical feature matrix")
print("First feature columns:", feature_table.columns[:10].tolist())
display(feature_table.head())

**05. Compare selected real features across activities**

We compare two RMS features describing back and thigh acceleration. These are source-scale descriptive features; no feature selection or classification has been performed.

In [ ]:
# 1. Attach source activity labels for visual analysis only.
#    The reusable feature-extraction function itself never sees the labels.
plot_data = feature_table[["back_x__rms", "thigh_x__rms"]].copy()
plot_data["activity"] = [ACTIVITY_NAMES[int(label)] for label in y_windows]

# 2. Display a scatter plot of two features coloured by observed activity.
fig, ax = plt.subplots(figsize=(9, 5))
for activity, subset in plot_data.groupby("activity", sort=True):
    ax.scatter(
        subset["back_x__rms"],
        subset["thigh_x__rms"],
        s=12, alpha=0.5, label=activity,
    )
ax.set(
    title="HARTH S023: RMS movement features by activity",
    xlabel="Lower-back X RMS (g)",
    ylabel="Thigh X RMS (g)",
)
ax.legend(title="Activity", fontsize=8, markerscale=1.5)
fig.tight_layout()
plt.show()

**06. Inspect feature distributions and class imbalance**

Descriptive features can be correlated and activity labels are imbalanced. A single-participant chart cannot establish how well any feature generalises to other people.

In [ ]:
# 1. Compare the mean RMS for each activity and show sample support.
comparison = (
    plot_data.groupby("activity", sort=True)
    .agg(
        retained_windows=("back_x__rms", "size"),
        mean_back_x_rms=("back_x__rms", "mean"),
        mean_thigh_x_rms=("thigh_x__rms", "mean"),
    )
    .sort_values("retained_windows", ascending=False)
)
display(comparison)

# 2. Plot per-activity variability for one feature on the original source scale.
grouped = list(plot_data.groupby("activity", sort=True))
fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(
    [group["back_x__rms"].to_numpy() for _, group in grouped],
    tick_labels=[activity for activity, _ in grouped],
    showfliers=False,
)
ax.set(
    title="HARTH S023: lower-back X RMS by activity",
    xlabel="Annotated activity",
    ylabel="RMS acceleration (g)",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()

**07. Quality gate: reproducible output, not a trained model**

Window/label correspondence is verified structurally. The next scientific phase requires multiple independent participants and group-disjoint validation before supervised classification.

In [ ]:
# 1. Confirm shape, column uniqueness and finite feature values.
assert len(feature_table) == len(X_windows) == len(y_windows)
assert feature_table.shape[1] == len(channel_names) * len(AVAILABLE_FEATURES)
assert feature_table.columns.is_unique
assert np.isfinite(feature_table.to_numpy()).all()
assert window_info["subject_id"].nunique() == 1

# 2. State the conclusions and restrictions explicitly.
print("Feature engineering: PASSED")
print("Samples:", len(feature_table), "| Features per sample:", feature_table.shape[1])
print("Independent subjects in this notebook: 1")
print("Feature standardisation fitted: NO")
print("Classifier trained: NO")
print("Subject-independent evaluation: NOT YET PERFORMED")